# Verify a design against Cameo requirements — in an existing system

Same requirements-driven verification as [the standalone version](workflow_log_cameo_requirements.ipynb), but it runs **against a system that already exists**. Nothing is created: the Cameo model, its extracted requirements, and the design parameter set are all found among the system's tracked files.

1. Resolve the system and classify its tracked files by role
2. Read the requirements the system already carries
3. Verify the tracked design against them and log the verdict
4. If anything fails, correct the design to a compliant value, re-verify, and log the pass
5. Attach the passing JUnit report to the design model

### What the system needs to contain

One configuration tracking these files. Names are matched case-insensitively:

| Tracked file | Role |
|---|---|
| `*.mdzip` | The Cameo model. Its newest `requirements.json` extraction supplies the limits. |
| `requirements.json` | Optional. When tracked directly it is used in preference to the extraction above. |
| `*design*.json` | The design parameter set under test. Substring set by `DESIGN_NAME_HINT`. |

### What this notebook writes to your system

It is not read-only. Be aware that it:

- Creates **workflow outputs and workflow log entries** on the system (additive)
- Snapshots the configuration and advances the **baseline tag** via `commit_changes` — a workflow log entry can only reference a configuration that is in the branch history
- With `APPLY_FIX = True`, pushes a **new revision of the tracked design file** and links a verified-report **artifact** to the design model

It never archives or deletes the system, its configuration, the Cameo model, or the design. The optional cleanup cell at the end archives only the log entries this run created.

### Prerequisites

Run these in a terminal from the cookbook root **before** starting this notebook:

```bash
uv sync --group dev
```

```bash
uv run python -m ipykernel install --user --name istari-client-cookbook --display-name "Python (istari-client-cookbook)"
```

Then pick the **Python (istari-client-cookbook)** kernel. Only the `dev` group is needed.

Also required:

- **Registry Service > 10.17.3** (2026-05 release or later)
- Credentials in [`samples/.env`](../.env): `ISTARI_REGISTRY_URL`, `ISTARI_PERSONAL_ACCESS_TOKEN`
- **Experimental features** enabled in the web app so the **Workflow log** tab is visible
- A `requirements.json` produced by `@istari:extract` with `tool_name="dassault_cameo"` — see [the Cameo extraction recipe](../cameo_extract_and_update_notebook%20-%20demo%20(sdk%20only).ipynb)

## 1. Connect

`Client` reads systems, tracked files, models, and artifacts; `V3Client` resolves resources and handles the workflow log.

In [ ]:
import copy
import json
import os
import re
from pathlib import Path

from dotenv import load_dotenv

from istari_helpers import commit_changes, find_configuration, find_system_by_name

from istari_digital_client import Client, Configuration, V3Client
from istari_digital_client.v3.models import WorkflowLogEntryCreateDto

load_dotenv("../.env")
REGISTRY_URL = os.environ["ISTARI_REGISTRY_URL"]

config = Configuration(
    registry_url=REGISTRY_URL,
    registry_auth_token=os.environ["ISTARI_PERSONAL_ACCESS_TOKEN"],
)
client = Client(config)   # systems, tracked files, models, artifacts
v3 = V3Client(config)     # resources + workflow log

UI_URL = REGISTRY_URL.rstrip("/").replace("//fileservice-v2.", "//")

print("Registry:", client.check_compatibility().server_version)
print("Web app:", UI_URL)

## 2. Resolve the system and classify its tracked files

Each tracked file carries a `resource_id`, so one `get_resource` call per file gives its name and type — no instance-wide scanning. The three roles are assigned from the file names.

In [ ]:
SYSTEM_ID = None            # set this to skip the name lookup
SYSTEM_NAME = "HVMC"        # used only when SYSTEM_ID is None
CONFIG_NAME = None          # None -> the system's first configuration
DESIGN_NAME_HINT = "design"  # substring identifying the design parameter file
APPLY_FIX = True            # write a corrected design revision when checks fail

if SYSTEM_ID is None:
    system = find_system_by_name(client, SYSTEM_NAME)
    if system is None:
        raise RuntimeError(
            f"No active system named {SYSTEM_NAME!r}. Set SYSTEM_ID, or correct SYSTEM_NAME."
        )
else:
    system = client.get_system(SYSTEM_ID)

SYSTEM_ID = system.id

if CONFIG_NAME is None:
    configurations = client.list_system_configurations(SYSTEM_ID, size=100).items
    if not configurations:
        raise RuntimeError(f"System {system.name!r} has no configurations.")
    configuration = configurations[0]
else:
    configuration = find_configuration(client, SYSTEM_ID, CONFIG_NAME)
    if configuration is None:
        raise RuntimeError(f"System {system.name!r} has no {CONFIG_NAME!r} configuration.")

CONFIG_ID = configuration.id
print(f"System:        {system.name}  ({SYSTEM_ID})")
print(f"Configuration: {configuration.name}  ({CONFIG_ID})")
print(f"               {UI_URL}/systems/{SYSTEM_ID}\n")

roles = {"cameo_model": None, "requirements": None, "design": None}

print(f"{'tracked file':<40}{'type':<10}role")
for tracked in client.list_tracked_files(configuration_id=CONFIG_ID, size=100).items:
    resource = v3.get_resource(resource_id=tracked.resource_id)
    name = resource.name or ""
    role = ""
    if name.lower().endswith(".mdzip"):
        role = "cameo_model"
    elif name.lower() == "requirements.json":
        role = "requirements"
    elif DESIGN_NAME_HINT.lower() in name.lower():
        role = "design"
    if role and roles[role] is None:
        roles[role] = (tracked, resource)
    kind = str(resource.resource_type).rsplit(".", 1)[-1].lower()
    print(f"  {name:<38}{kind:<10}{role or '-'}")

if roles["design"] is None:
    raise RuntimeError(
        f"No tracked file matching {DESIGN_NAME_HINT!r} on configuration {configuration.name!r}. "
        "Set DESIGN_NAME_HINT to the substring that identifies your design parameter file."
    )
if roles["requirements"] is None and roles["cameo_model"] is None:
    raise RuntimeError(
        "The configuration tracks neither a requirements.json nor a .mdzip Cameo model, so "
        "there are no requirements to verify against."
    )

## 3. Read the requirements the system carries

A tracked `requirements.json` is used directly. Otherwise the newest `requirements.json` artifact on the tracked Cameo model is read — the product of `@istari:extract`.

`parse_limits` pulls the numeric bounds out of the SysML requirement text. Three phrasings are recognised:

| Requirement text | Parsed as |
|---|---|
| *"shall be 9 inches"* | exact nominal, `= 9 in` |
| *"shall not exceed 110°C"* / *"up to 70°C"* | upper bound, `<= 110 °C` |
| *"shall be between 26 mm and 30 mm"* | range, `26-30 mm` |

Units normalise to `in`, `°C`, and `mm`. Requirements with no number — narrative parents, empty placeholders — are skipped.

In [ ]:
_UNIT = r"(?:inches|inch|in|mm|°\s*C|degC|C)"
_RANGE_RE = re.compile(
    rf"between\s+(?P<low>[\d.]+)\s*(?:{_UNIT})?\s+and\s+(?P<high>[\d.]+)\s*(?P<unit>{_UNIT})\b",
    re.IGNORECASE)
_MAX_RE = re.compile(
    rf"(?:shall not exceed|up to|no more than|at most)\s+(?P<high>[\d.]+)\s*(?P<unit>{_UNIT})\b",
    re.IGNORECASE)
_EXACT_RE = re.compile(
    rf"shall be\s+(?P<value>[\d.]+)\s*(?P<unit>{_UNIT})\b",
    re.IGNORECASE)
TOLERANCE = 1e-9  # exact-nominal requirements compare with a float epsilon


def normalize_unit(raw):
    """Collapse the spellings Cameo authors use into one unit label."""
    unit = raw.replace(" ", "").lower()
    if unit in ("inches", "inch", "in"):
        return "in"
    if unit in ("°c", "degc", "c"):
        return "°C"
    return unit


def parse_limits(text):
    """Return (low, high, unit) from SysML requirement text, or None if not numeric.

    Either bound may be None, meaning unbounded on that side; low == high means the
    requirement states an exact nominal value.
    """
    match = _RANGE_RE.search(text)
    if match:
        return float(match["low"]), float(match["high"]), normalize_unit(match["unit"])
    match = _MAX_RE.search(text)
    if match:
        return None, float(match["high"]), normalize_unit(match["unit"])
    match = _EXACT_RE.search(text)
    if match:
        value = float(match["value"])
        return value, value, normalize_unit(match["unit"])
    return None


def describe_limits(low, high, unit):
    """Readable window for a parsed requirement."""
    if low is None:
        return f"<= {high:g} {unit}"
    if low == high:
        return f"= {low:g} {unit}"
    return f"{low:g}-{high:g} {unit}"


if roles["requirements"] is not None:
    tracked, resource = roles["requirements"]
    revision = client.get_file(tracked.file_id).revisions[-1]
    REQUIREMENTS_BYTES = revision.read_bytes()
    REQ_SOURCE = {
        "origin": "tracked requirements.json",
        "cameo_model": roles["cameo_model"][1].name if roles["cameo_model"] else None,
        "resource_id": resource.resource_id,
        "revision_id": revision.id,
    }
else:
    tracked, resource = roles["cameo_model"]
    artifacts = [
        a for a in client.list_model_artifacts(resource.resource_id, size=50).items
        if a.file and a.file.name == "requirements.json"
    ]
    if not artifacts:
        raise RuntimeError(
            f"Tracked Cameo model {resource.name!r} has no requirements.json artifact. Run "
            '@istari:extract with tool_name="dassault_cameo" on it first.'
        )
    artifact = artifacts[-1]
    revision = artifact.file.revisions[-1]
    REQUIREMENTS_BYTES = revision.read_bytes()
    REQ_SOURCE = {
        "origin": f"extraction on {resource.name}",
        "cameo_model": resource.name,
        "resource_id": artifact.id,
        "revision_id": revision.id,
    }

requirements = json.loads(REQUIREMENTS_BYTES.decode("utf-8"))

print(f"Requirements from: {REQ_SOURCE['origin']}")
print(f"  resource {REQ_SOURCE['resource_id']}, revision {REQ_SOURCE['revision_id']}")
print(f"{len(requirements)} requirements; numerically bounded ones:\n")
for req in requirements:
    limits = parse_limits(req.get("text") or "")
    if limits:
        name = (req.get("name") or "").strip()
        print(f"  [{req['req_id']:<9}] {name:<48} {describe_limits(*limits)}")

## 4. Verify the tracked design

`PARAM_FOR_REQ` is the verification matrix: which design parameter satisfies which Cameo requirement. Keys are matched with surrounding whitespace stripped, because `"Extended Requirement Coolant Temperature "` ships from Cameo with a trailing space.

The design is read from the system, not written here — whatever revision the configuration currently tracks is what gets judged.

Each run writes four output files:

| File | Contents |
|---|---|
| `verification-results.xml` | JUnit report — one `testcase` per requirement, a `failure` element for each violation. The artifact attached to the model in §7, and the file a CI dashboard would read. |
| `results.json` | Machine-readable detail: every check with its `req_id`, limits, value, and the requirements source. |
| `report.txt` | The same table printed below. |
| `requirements.json` | The exact requirement set this verdict was judged against. |

In [ ]:
from xml.sax.saxutils import escape

PARAM_FOR_REQ = {
    # Physical Characteristics :: Coldplate Dimensions
    "Extended Requirement Coldplate Length": ("COLDPLATE_1", "length_in"),
    "Extended Requirement Coldplate Width": ("COLDPLATE_1", "width_in"),
    "Extended Requirement Coldplate Height": ("COLDPLATE_1", "height_in"),
    # Environmental Conditions :: Natural Environment :: Temperture
    "Extended Requirement Coolant Temperature": ("THERMAL", "coolant_temp_c"),
    "Extended Requirement Component Temperature": ("THERMAL", "component_temp_c"),
    "Extended Requirement Power Module Temperature": ("THERMAL", "power_module_temp_c"),
}

design_tracked, design_resource = roles["design"]
DESIGN_RESOURCE_ID = design_resource.resource_id
DESIGN_FILE_ID = design_tracked.file_id
DESIGN_NAME = design_resource.name or "design.json"
DESIGN_STEM = Path(DESIGN_NAME).stem

work = Path("_insystem_run")
work.mkdir(exist_ok=True)
local_design = work / DESIGN_NAME


def failures_line(checks):
    """e.g. "1 of 6 requirements failed"."""
    failed = sum(not check["passed"] for check in checks)
    return f"{failed} of {len(checks)} requirements failed"


def write_junit(checks, path, src_name):
    """Write the verification results as a JUnit XML report; returns the path.

    One testcase per verified requirement, named by its Cameo req_id, so a CI
    dashboard reports failures against the requirement that was violated.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    failures = sum(not check["passed"] for check in checks)
    cases = []
    for check in checks:
        body = (
            f'\n    <failure message="{escape(check["detail"])}"/>\n  '
            if not check["passed"] else ""
        )
        case_name = f'{check["req_id"]} {check["requirement"]}'
        cases.append(
            f'  <testcase classname="verification.{escape(src_name)}" '
            f'name="{escape(case_name)}">{body}</testcase>'
        )

    path.write_text(
        '<?xml version="1.0" encoding="utf-8"?>\n'
        f'<testsuite name="requirements-verification [{escape(src_name)}]" '
        f'tests="{len(checks)}" failures="{failures}" errors="0">\n'
        + "\n".join(cases)
        + "\n</testsuite>\n"
    )
    return path


def run_checks(design_bytes, out_dir):
    """Verify every mapped requirement; return (verdict, checks, output paths)."""
    values = json.loads(design_bytes)
    checks = []

    for req in requirements:
        target = PARAM_FOR_REQ.get((req.get("name") or "").strip())
        limits = parse_limits(req.get("text") or "")
        if target is None or limits is None:
            continue  # narrative or unmapped requirement - not machine-verifiable here

        part, parameter = target
        if part not in values or parameter not in values[part]:
            raise RuntimeError(
                f"Requirement {req['req_id']} maps to {part}.{parameter}, which the tracked "
                f"design {design_resource.name!r} does not contain. Keys present: "
                f"{ {k: sorted(v) for k, v in values.items()} }"
            )
        value = values[part][parameter]
        low, high, unit = limits
        passed = ((low is None or value >= low - TOLERANCE)
                  and (high is None or value <= high + TOLERANCE))

        checks.append({
            "req_id": req["req_id"],
            "requirement": (req.get("name") or "").strip(),
            "parameter": f"{part}.{parameter}",
            "value": value,
            "unit": unit,
            "limits": [low, high],
            "passed": passed,
            "detail": f"{value:g} {unit} against {describe_limits(low, high, unit)}",
        })

    if not checks:
        raise RuntimeError(
            "No requirement matched PARAM_FOR_REQ. Extracted names: "
            f"{[(r.get('name') or '').strip() for r in requirements]}"
        )

    verdict = "SUCCESS" if all(check["passed"] for check in checks) else "FAILED"

    out_dir.mkdir(parents=True, exist_ok=True)
    results_path = out_dir / "results.json"
    report_path = out_dir / "report.txt"
    requirements_path = out_dir / "requirements.json"
    junit_path = write_junit(checks, out_dir / "verification-results.xml", DESIGN_STEM)

    results_path.write_text(json.dumps({
        "verdict": verdict,
        "system_id": SYSTEM_ID,
        "configuration_id": CONFIG_ID,
        "design": {"resource_id": DESIGN_RESOURCE_ID, "name": DESIGN_NAME},
        "requirements_source": REQ_SOURCE,
        "checks": checks,
    }, indent=2))
    report_path.write_text(
        "".join(
            f"{'PASS' if check['passed'] else 'FAIL'}  [{check['req_id']:<9}] "
            f"{check['parameter']:<30}{check['detail']}\n"
            for check in checks
        )
    )
    requirements_path.write_bytes(REQUIREMENTS_BYTES)

    print(report_path.read_text(), end="")
    print("Verdict:", verdict, f"({failures_line(checks)})")
    return verdict, checks, [junit_path, results_path, report_path, requirements_path]


design_bytes = client.get_file(file_id=DESIGN_FILE_ID).revisions[-1].read_bytes()
print(f"Design under test: {DESIGN_NAME} (resource {DESIGN_RESOURCE_ID})\n")
verdict, checks, output_paths = run_checks(design_bytes, work / "iter-1")

## 5. Log the verdict on the system

`create_workflow_output` registers each result file against the system; `create_workflow_log_entry` ties the title, `status`, `configuration_id`, and output IDs into one durable record.

`commit_changes` runs first because an entry can only reference a configuration that is in the branch history. On an already-committed system the snapshot is a no-op and only the baseline tag moves to the newest snapshot for this configuration.

> **Pause here.** Open the system in the web app → **Workflow log** tab and preview the attached outputs, including the requirement set the run was judged against.

In [ ]:
commit_changes(client, SYSTEM_ID, CONFIG_ID)


def log_run(title, verdict, paths):
    """Upload each output file, then record one workflow log entry."""
    output_ids = [v3.create_workflow_output(system_id=SYSTEM_ID, path=p).id for p in paths]
    entry = v3.create_workflow_log_entry(
        system_id=SYSTEM_ID,
        workflow_log_entry_create_dto=WorkflowLogEntryCreateDto(
            title=title,
            status=verdict,  # SUCCESS | FAILED | UNSPECIFIED
            configuration_id=CONFIG_ID,
            workflow_output_ids=output_ids,
        ),
    )
    print(f"{entry.status}  {title}  ({len(output_ids)} outputs, entry {entry.id})")
    return entry


entry1 = log_run("Requirements verification - as tracked", verdict, output_paths)
entry2 = None
print("Workflow log tab:", f"{UI_URL}/systems/{SYSTEM_ID}")

## 6. Correct the design, re-verify, and log the pass

Only runs when something failed and `APPLY_FIX` is `True`. Each failing parameter is moved inside its own requirement window — the stated nominal for an exact requirement, the midpoint of a range, or 5 % under a ceiling — so the correction comes from the requirement rather than from a number typed here.

The corrected values are pushed as a **new revision of the tracked design file**, exactly as a drag-and-drop in the web app would, and the configuration is committed again so the second entry can reference it.

In [ ]:
MARGIN = 0.05  # how far inside a one-sided bound to land


def compliant_value(low, high):
    """A value inside the requirement window: nominal, midpoint, or just under a bound."""
    if low is None:
        return round(high * (1 - MARGIN), 3)
    if high is None:
        return round(low * (1 + MARGIN), 3)
    if low == high:
        return low
    return round((low + high) / 2, 3)


def propose_fix(values, checks):
    """Return a copy of *values* with every failing parameter moved inside its window."""
    fixed = copy.deepcopy(values)
    changes = []
    for check in checks:
        if check["passed"]:
            continue
        part, parameter = check["parameter"].split(".", 1)
        new_value = compliant_value(*check["limits"])
        fixed[part][parameter] = new_value
        changes.append((check["req_id"], check["parameter"], check["value"],
                        new_value, check["unit"]))
    return fixed, changes


if verdict == "SUCCESS":
    print("Design already satisfies every mapped requirement - nothing to correct.")
elif not APPLY_FIX:
    print("APPLY_FIX is False - leaving the tracked design untouched.")
else:
    fixed_values, changes = propose_fix(json.loads(design_bytes), checks)
    for req_id, parameter, old, new, unit in changes:
        print(f"  [{req_id}] {parameter}: {old:g} -> {new:g} {unit}")

    local_design.write_text(json.dumps(fixed_values, indent=2))
    client.update_model(
        model_id=DESIGN_RESOURCE_ID,
        path=local_design,
        description="Corrected to satisfy Cameo requirements",
        version_name="requirements-compliant",
    )
    commit_changes(client, SYSTEM_ID, CONFIG_ID)

    design_bytes = client.get_file(file_id=DESIGN_FILE_ID).revisions[-1].read_bytes()
    print()
    verdict, checks, output_paths = run_checks(design_bytes, work / "iter-2")
    entry2 = log_run("Requirements verification - after correction", verdict, output_paths)
    print("Review both entries:", f"{UI_URL}/systems/{SYSTEM_ID}")

## 7. Attach the verified report to the design model

Runs only when the design now passes. The artifact is the **JUnit XML report** — the same evidence a CI verification job would publish, and the file most downstream tooling expects. `create_resource(resource_type=artifact)` uploads it — the V3 Resources equivalent of `Client.add_artifact()` — and a `produces` revision relationship links the design model revision to it. Workflow outputs stay outside configuration snapshots; a model-linked artifact does not, so one more `commit_changes` puts it on the baseline snapshot.

In [ ]:
import shutil

from istari_digital_client.v3.models.new_revision_relationship_dto import NewRevisionRelationshipDto
from istari_digital_client.v3.models.resource_type_dto import ResourceTypeDto

if verdict != "SUCCESS":
    print(f"Verdict is {verdict} - not attaching a verified report.")
else:
    junit_report = next(p for p in output_paths if p.suffix == ".xml")
    verified = work / f"verified-test-results-for-{DESIGN_STEM}.xml"
    shutil.copyfile(junit_report, verified)

    artifact = v3.create_resource(
        path=verified,
        resource_type=ResourceTypeDto.ARTIFACT,
        display_name=verified.name,
        description=f"JUnit verification report - verified against {REQ_SOURCE['origin']}",
    )

    design_model_resource = v3.get_resource(resource_id=DESIGN_RESOURCE_ID)
    produces = next(t for t in v3.list_revision_relationship_types().items if t.name == "produces")
    v3.create_revision_relationship(
        new_revision_relationship_dto=NewRevisionRelationshipDto(
            relationship_type_id=produces.id,
            left_revision_id=design_model_resource.file_revision_id,
            right_revision_id=artifact.file_revision_id,
        ),
    )
    commit_changes(client, SYSTEM_ID, CONFIG_ID)

    print(f"Artifact {artifact.resource_id} linked to design {DESIGN_RESOURCE_ID} "
          f"via {produces.name}")
    print("Committed to the baseline snapshot")

## Recap

The system you started with now carries, on its own configuration:

- A workflow log entry for the design **as it was tracked**, and — when a correction was needed — a second one showing the pass
- Every result file from each run — JUnit XML, JSON detail, text report, and the exact requirement revision the verdict was judged against
- The passing **JUnit report** (`verification-results.xml`) as an artifact on the design model via a `produces` relationship

No limit was written down here. The requirements came out of the system's own Cameo extraction, each result carries the `req_id` it satisfies, and the correction in §6 was derived from the requirement window — so re-extracting after a requirement change re-judges the same design with no code edit.

### Learn more

- [External workflow logs](https://docs.istaridigital.com/developers/SDK/v3/03-workflow-logs) - `create_workflow_output`, `create_workflow_log_entry`, and listing entries
- [Cameo requirements extraction + tag update](../cameo_extract_and_update_notebook%20-%20demo%20(sdk%20only).ipynb) - producing `requirements.json`, and writing values back with `@istari:update_tags`
- [Standalone version](workflow_log_cameo_requirements.ipynb) - same verification, creating its own throwaway system

## Cleanup (optional)

Archives only the workflow log entries this run created. The system, its configuration, the Cameo model, the design, and every snapshot are left as they were. Archiving an entry is reversible with `v3.restore_workflow_log_entry`.

In [ ]:
for entry in [e for e in (entry1, entry2) if e is not None]:
    v3.archive_workflow_log_entry(system_id=SYSTEM_ID, entry_id=entry.id)
    print("Archived workflow log entry", entry.id)